<a href="https://colab.research.google.com/github/JakeFRCSE/refusal-direction-reproduction/blob/main/notebooks/refusal_direction_reproduction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Refusal Direction in LLMs

Goal
Reproduce the experiment of Section 3 of
["Refusal in Language Models Is Mediated by a Single Direction"](https://arxiv.org/pdf/2406.11717)

Main steps
1. Extract residual directions
2. Compute candidate directions
3. Select optimal direction following the paper
4. Run intervention experiments
5. Extend experiments

# 0. Loading Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
from pathlib import Path
import json
from itertools import combinations

In [ ]:
DATASET_DIR = Path("/content/drive/MyDrive/research/research-cycle1/throw-away-projects/refusal/dataset/processed")

SPLIT_SIZES = {
    "train": 128,
    "val": 32,
    "test": 100,
}

CATEGORIES = ["harmful", "harmless"]


def load_dataset(file_name: str, dir_path: Path = DATASET_DIR):
    with open(dir_path / file_name, "r", encoding="utf-8") as f:
        return json.load(f)


def load_split(category: str, split: str, dir_path: Path = DATASET_DIR):
    file_name = f"{category}_{split}.json"
    data = load_dataset(file_name, dir_path)
    return data[:SPLIT_SIZES[split]]


raw_datasets = {
    category: {
        split: load_split(category, split)
        for split in SPLIT_SIZES
    }
    for category in CATEGORIES
}

In [ ]:
def get_instruction_set(dataset):
    return {x["instruction"] for x in dataset}

def get_pairwise_overlaps(dataset_dict):
    instruction_sets = {
        name: get_instruction_set(dataset)
        for name, dataset in dataset_dict.items()
    }

    overlaps = {}
    for a, b in combinations(instruction_sets.keys(), 2):
        overlaps[(a, b)] = instruction_sets[a] & instruction_sets[b]
    return overlaps

In [ ]:
overlap_test_datasets = {
    f"{category}_{split}": raw_datasets[category][split]
    for category, splits in raw_datasets.items()
    for split in splits
}

overlaps = get_pairwise_overlaps(overlap_test_datasets)

for (a, b), overlap in overlaps.items():
    print(f"{a} vs {b}: {len(overlap)}")

# 1. Computing and selecting $r$

In [ ]:
! pip install transformer_lens

## 1-0. Load Model & Construct Dataset

In [ ]:
from transformer_lens import HookedTransformer, utils
import torch
import torch.nn.functional as F
from datasets import Dataset
from torch.utils.data import DataLoader
import functools
from tqdm.auto import tqdm

In [ ]:
MODEL_CONFIGS = {
    "google/gemma-2b-it": {
        "prompt_template": "<start_of_turn>user\n{x}<end_of_turn>\n<start_of_turn>model\n",
        "post_instruction_text": "<end_of_turn>\n<start_of_turn>model\n",
        "refusal_token_set" : {235285},
    },
    "Qwen/Qwen-1_8B-Chat": {
        "prompt_template": "<|im_start|>user\n{x}<|im_end|>\n<|im_start|>assistant\n",
        "post_instruction_text": "<|im_end|>\n<|im_start|>assistant\n",
        "refusal_token_set" : {40, 2121},
    },
}


def get_model_config(model_name: str):
    if model_name not in MODEL_CONFIGS:
        raise ValueError(f"Unsupported model: {model_name}")
    return MODEL_CONFIGS[model_name]


device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "google/gemma-2b-it"

model_config = get_model_config(model_name)

model = HookedTransformer.from_pretrained(model_name)
prompt_template = model_config["prompt_template"]
post_instruction_text = model_config["post_instruction_text"]
refusal_token_set = model_config["refusal_token_set"]

In [ ]:
def make_collate_fn(model, prompt_template, post_instruction_text, device):
    pad_id = model.tokenizer.pad_token_id
    if pad_id is None:
        pad_id = model.tokenizer.eos_token_id

    post_instruction_tokens = model.to_tokens(post_instruction_text)[:, 1:].to(device)
    post_instruction_length = post_instruction_tokens.size(1)

    def collate(batch):
        if isinstance(batch[0], dict):
            instructions = [x["instruction"] for x in batch]
        else:
            instructions = list(batch)

        prompts = [prompt_template.format(x=ins) for ins in instructions]
        tokenized_prompts = [model.to_tokens(prompt).squeeze(0) for prompt in prompts]

        lengths = torch.tensor(
            [tokens.numel() for tokens in tokenized_prompts],
            device=device,
        )
        max_len = int(lengths.max().item())

        batch_size = len(tokenized_prompts)
        input_ids = torch.full((batch_size, max_len), pad_id, device=device, dtype=torch.long)
        attention_mask = torch.zeros((batch_size, max_len), device=device, dtype=torch.bool)

        for i, tokens in enumerate(tokenized_prompts):
            seq_len = tokens.numel()
            input_ids[i, :seq_len] = tokens
            attention_mask[i, :seq_len] = True

        cache_start = (lengths - post_instruction_length).to(torch.long)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "lengths": lengths,
            "cache_start": cache_start,
            "post_instruction_length": post_instruction_length,
            "instructions": instructions,
            "prompts": prompts,
        }

    return collate

In [ ]:
batch_size = 8

collate_fn = make_collate_fn(
    model=model,
    prompt_template=prompt_template,
    post_instruction_text=post_instruction_text,
    device=device,
)

datasets = {
    category: {
        split: Dataset.from_list(data)
        for split, data in split_dict.items()
    }
    for category, split_dict in raw_datasets.items()
}

dataloaders = {
    category: {
        split: DataLoader(
            datasets[category][split],
            batch_size=batch_size,
            shuffle=False,
            collate_fn=collate_fn,
        )
        for split in datasets[category]
    }
    for category in datasets
}

dl_hf_train = dataloaders["harmful"]["train"]
dl_hf_val = dataloaders["harmful"]["val"]
dl_hf_test = dataloaders["harmful"]["test"]

dl_hl_train = dataloaders["harmless"]["train"]
dl_hl_val = dataloaders["harmless"]["val"]
dl_hl_test = dataloaders["harmless"]["test"]

## 1-1. Load Utility Functions

In [ ]:
def get_last_token_logits(logits, lengths):
    batch_size, seq_len, _ = logits.shape  # [B, S, V]
    last_token_pos = (lengths.to(logits.device) - 1).clamp(0, seq_len - 1)
    return logits[torch.arange(batch_size, device=logits.device), last_token_pos]  # [B, V]


def build_ablation_hook(vector):
    def ablate_direction(activation, hook):
        direction = vector / (vector.norm() + 1e-8)
        projection = torch.einsum("bsd,d->bs", activation, direction)
        return activation - projection[..., None] * direction

    return ablate_direction


def build_addition_hook(vector):
    def add_direction(activation, hook):
        return activation + vector[None, None, :]

    return add_direction


def get_ablation_hook_points(model):
    return [
        *(utils.get_act_name("resid_pre", layer_idx) for layer_idx in range(model.cfg.n_layers)),
        *(utils.get_act_name("resid_mid", layer_idx) for layer_idx in range(model.cfg.n_layers)),
    ]

def compute_refusal_logodds(logits, refusal_token_ids):
    refusal_token_ids = torch.tensor(list(refusal_token_ids), device=logits.device)

    if logits.ndim == 3:
        logits = logits.squeeze(1)

    refusal_prob = logits.softmax(dim=-1)[:, refusal_token_ids].sum(dim=-1)

    eps = 1e-8
    refusal_prob = refusal_prob.clamp(eps, 1 - eps)

    return torch.log(refusal_prob) - torch.log(1 - refusal_prob)

@torch.no_grad()
def compute_refusal_scores_with_hooks(
    dataloader,
    model,
    refusal_token_ids,
    fwd_hooks,
):
    batch_scores = []

    for batch in dataloader:
        input_ids = batch["input_ids"].to(model.cfg.device)
        lengths = batch["lengths"].to(model.cfg.device)

        logits = model.run_with_hooks(
            input_ids,
            return_type="logits",
            fwd_hooks=fwd_hooks,
        )
        logits_last = get_last_token_logits(logits, lengths)
        refusal_scores = compute_refusal_logodds(logits_last, refusal_token_ids)

        batch_scores.append(refusal_scores.detach().cpu())

        del logits
        torch.cuda.empty_cache()

    return torch.cat(batch_scores, dim=0)

In [ ]:
@torch.no_grad()
def compute_mean_resid_pre_span(dataloader, model):
    all_batch_spans = []

    for batch in dataloader:
        input_ids = batch["input_ids"]                         # [B, S]
        span_start = batch["cache_start"].to(input_ids.device) # [B]
        span_length = int(batch["post_instruction_length"])
        batch_size, seq_len = input_ids.shape

        _, cache = model.run_with_cache(
            input_ids,
            return_type=None,
            return_cache_object=True,
        )

        span_positions = (
            span_start[:, None]
            + torch.arange(span_length, device=input_ids.device)[None, :]
        )
        span_positions = span_positions.clamp(max=seq_len - 1)

        d_model = model.cfg.d_model
        gather_index = span_positions[:, :, None].expand(batch_size, span_length, d_model)

        layer_spans = []
        for layer in range(model.cfg.n_layers):
            resid_pre = cache["resid_pre", layer]
            resid_span = resid_pre.gather(1, gather_index)
            layer_spans.append(resid_span.unsqueeze(1))

        batch_span_tensor = torch.cat(layer_spans, dim=1)  # [B, L, K, D]
        all_batch_spans.append(batch_span_tensor)

        del cache

    all_spans = torch.cat(all_batch_spans, dim=0)          # [N, L, K, D]
    return all_spans.mean(dim=0)                           # [L, K, D]

In [ ]:
@torch.no_grad()
def compute_bypass_score(val_dataloader, ablation_vectors, model, refusal_token_ids):
    n_layers, n_positions, _ = ablation_vectors.shape
    bypass_scores = torch.zeros((n_layers, n_positions), dtype=torch.float32)

    hook_points = get_ablation_hook_points(model)

    for layer_idx in range(n_layers):
        for pos_idx in range(n_positions):
            hook_fn = build_ablation_hook(ablation_vectors[layer_idx, pos_idx])
            fwd_hooks = [(hook_name, hook_fn) for hook_name in hook_points]

            refusal_scores = compute_refusal_scores_with_hooks(
                val_dataloader,
                model,
                refusal_token_ids,
                fwd_hooks,
            )
            bypass_scores[layer_idx, pos_idx] = refusal_scores.mean().item()

    return bypass_scores

@torch.no_grad()
def compute_induce_score(val_dataloader, add_vectors, model, refusal_token_ids):
    n_layers, n_positions, _ = add_vectors.shape
    induce_scores = torch.zeros((n_layers, n_positions), dtype=torch.float32)

    for layer_idx in range(n_layers):
        for pos_idx in range(n_positions):
            hook_fn = build_addition_hook(add_vectors[layer_idx, pos_idx])
            fwd_hooks = [(utils.get_act_name("resid_pre", layer_idx), hook_fn)]

            refusal_scores = compute_refusal_scores_with_hooks(
                val_dataloader,
                model,
                refusal_token_ids,
                fwd_hooks,
            )
            induce_scores[layer_idx, pos_idx] = refusal_scores.mean().item()

    return induce_scores

def kl_divergence_from_logits(baseline_logits, perturbed_logits):
    baseline_logits = baseline_logits.to(torch.float64)
    perturbed_logits = perturbed_logits.to(torch.float64)

    baseline_probs = F.softmax(baseline_logits, dim=-1)
    baseline_log_probs = F.log_softmax(baseline_logits, dim=-1)
    perturbed_log_probs = F.log_softmax(perturbed_logits, dim=-1)

    kl_values = (baseline_probs * (baseline_log_probs - perturbed_log_probs)).sum(dim=-1)
    return kl_values.mean()


@torch.no_grad()
def compute_kl_score(val_dataloader, ablation_vectors, model):
    n_layers, n_positions, _ = ablation_vectors.shape
    kl_scores = torch.zeros((n_layers, n_positions), dtype=torch.float32)

    hook_points = get_ablation_hook_points(model)

    baseline_last_logits_batches = []
    input_batches = []
    length_batches = []

    for batch in val_dataloader:
        input_ids = batch["input_ids"].to(model.cfg.device)
        lengths = batch["lengths"].to(model.cfg.device)

        baseline_logits = model(input_ids, return_type="logits")
        baseline_last_logits = get_last_token_logits(baseline_logits, lengths)

        input_batches.append(input_ids)
        length_batches.append(lengths)
        baseline_last_logits_batches.append(baseline_last_logits.detach())

        del baseline_logits
        torch.cuda.empty_cache()

    for layer_idx in range(n_layers):
        for pos_idx in range(n_positions):
            hook_fn = build_ablation_hook(ablation_vectors[layer_idx, pos_idx])
            fwd_hooks = [(hook_name, hook_fn) for hook_name in hook_points]

            kl_sum = 0.0

            for input_ids, lengths, baseline_last_logits in zip(
                input_batches,
                length_batches,
                baseline_last_logits_batches,
            ):
                perturbed_logits = model.run_with_hooks(
                    input_ids,
                    return_type="logits",
                    fwd_hooks=fwd_hooks,
                )
                perturbed_last_logits = get_last_token_logits(perturbed_logits, lengths)

                kl_sum += kl_divergence_from_logits(
                    baseline_last_logits,
                    perturbed_last_logits,
                ).item()

                del perturbed_logits
                torch.cuda.empty_cache()

            kl_scores[layer_idx, pos_idx] = kl_sum / max(len(input_batches), 1)

    return kl_scores

In [ ]:
def evaluate_refusal_case(logits, refusal_token_ids):
    refusal_logodds = compute_refusal_logodds(logits, refusal_token_ids)
    return refusal_logodds >= 0

@torch.no_grad()
def compute_induce_cases(test_dataloader, add_vector, model, refusal_token_ids, layer_idx):
    hook_fn = build_addition_hook(add_vector)
    fwd_hooks = [(utils.get_act_name("resid_pre", layer_idx), hook_fn)]

    refusal_scores = compute_refusal_scores_with_hooks(
        test_dataloader,
        model,
        refusal_token_ids,
        fwd_hooks,
    )
    return (refusal_scores >= 0).tolist()

@torch.no_grad()
def compute_bypass_cases(test_dataloader, ablation_vector, model, refusal_token_ids):
    hook_fn = build_ablation_hook(ablation_vector)
    hook_points = get_ablation_hook_points(model)
    fwd_hooks = [(hook_name, hook_fn) for hook_name in hook_points]

    refusal_scores = compute_refusal_scores_with_hooks(
        test_dataloader,
        model,
        refusal_token_ids,
        fwd_hooks,
    )
    return (refusal_scores >= 0).tolist()

## 1-2. Selecting $r_{i^*}^{l^*}$

In [ ]:
harmful_mean_resid_train = compute_mean_resid_pre_span(dl_hf_train, model)
harmless_mean_resid_train = compute_mean_resid_pre_span(dl_hl_train, model)

refusal_direction_train = harmful_mean_resid_train - harmless_mean_resid_train

ablate_bypass_score = compute_bypass_score(
    dl_hf_val,
    refusal_direction_train,
    model,
    refusal_token_set,
)

add_induce_score = compute_induce_score(
    dl_hl_val,
    refusal_direction_train,
    model,
    refusal_token_set,
)

kl_score = compute_kl_score(
    dl_hl_val,
    refusal_direction_train,
    model,
)

candidate_mask = (kl_score < 0.1) & (add_induce_score > 0)

masked_bypass_scores = torch.where(
    candidate_mask,
    ablate_bypass_score,
    torch.full_like(ablate_bypass_score, float("inf")),
)

flat_best_idx = torch.argmin(masked_bypass_scores)
best_layer_idx, best_pos_idx = torch.unravel_index(
    flat_best_idx,
    ablate_bypass_score.shape,
)

print(f"layer: {best_layer_idx}")
print(f"token index: {best_pos_idx}")

print(f"add_induce_score: {add_induce_score[best_layer_idx, best_pos_idx]}")
print(f"ablate_bypass_score: {ablate_bypass_score[best_layer_idx, best_pos_idx]}")
print(f"kl_score: {kl_score[best_layer_idx, best_pos_idx]}")

refusal_direction = refusal_direction_train[best_layer_idx, best_pos_idx]

## 1-3. Executing Intervention

In [ ]:
# Should be high
induced_cases = compute_induce_cases(
    dl_hl_test,
    refusal_direction,
    model,
    refusal_token_set,
    best_layer_idx,
)

# Should be low
baseline_induced_cases = compute_induce_cases(
    dl_hl_test,
    torch.zeros_like(refusal_direction),
    model,
    refusal_token_set,
    best_layer_idx,
)

# Should be low
bypassed_cases = compute_bypass_cases(
    dl_hf_test,
    refusal_direction,
    model,
    refusal_token_set,
)

# Should be high
baseline_bypassed_cases = compute_bypass_cases(
    dl_hf_test,
    torch.zeros_like(refusal_direction),
    model,
    refusal_token_set,
)

# 2. Plotting Results

In [ ]:
import matplotlib.pyplot as plt

induced_refusal_count = torch.tensor(induced_cases).sum().item()
baseline_induced_refusal_count = torch.tensor(baseline_induced_cases).sum().item()

bypassed_refusal_count = torch.tensor(bypassed_cases).sum().item()
baseline_bypassed_refusal_count = torch.tensor(baseline_bypassed_cases).sum().item()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

plt.suptitle(f"{model_name.split("/")[1]} refusal fig 3 reproduce")

axes[0].bar(
    ["baseline", "induced"],
    [baseline_induced_refusal_count, induced_refusal_count],
)
axes[0].set_title("Harmless prompts")
axes[0].set_ylabel("Refusal score")

axes[1].bar(
    ["baseline", "bypassed"],
    [baseline_bypassed_refusal_count, bypassed_refusal_count],
)
axes[1].set_title("Harmful prompts")


plt.tight_layout()

BASE_DIR = Path("/content/drive/MyDrive/research/research-cycle1/throw-away-projects/refusal/")
SAVE_DIR = BASE_DIR / "results"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

save_path = SAVE_DIR / f"{model_name}-refusal-fig3-reproduce.png"
save_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(save_path, dpi=300, bbox_inches="tight")

plt.show()


Result Interpretation

Harmless prompts:
Adding the refusal direction induces refusal behavior.

Harmful prompts:
Ablating the refusal direction suppresses refusal behavior.

This supports the hypothesis that refusal is mediated by a single direction in the residual stream.

# 3. Extended Experiments

TL;DR

I extended the main experiment.

Key findings:

1. The effect localizes around layer 9 in post-instruction tokens.
2. No single attention head explains the behavior.

In [ ]:
import torch
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm
from transformer_lens import utils

In [ ]:
BASE_DIR = Path("/content/drive/MyDrive/research/research-cycle1/throw-away-projects/refusal/")
SAVE_DIR = BASE_DIR / "results"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

## 3-1. post-instruction token localized ablation / addition

**Observation**

Refusal direction $𝑟$ globally controls behaviour.

**Question**

Which **layer / post-instruction token** actually uses this direction?

**Result**

- Strongest effect around **layer 9.**

- Indicates refusal representation is **formed or used around layer 9,** not uniformly across the network.

In [ ]:
@torch.no_grad()
def compute_localized_scores(
    dataloader,
    model,
    refusal_token_ids,
    vector,
    layer_idx,
    rel_pos,
    mode,   # "ablate" or "add"
):
    assert mode in {"ablate", "add"}

    def localized_hook(activation, hook):
        activation = activation.clone()
        batch_size, seq_len, d_model = activation.shape

        token_positions = batch_cache_start + rel_pos
        batch_idx = torch.arange(batch_size, device=activation.device)
        valid = (token_positions >= 0) & (token_positions < seq_len)

        if valid.any():
            if mode == "ablate":
                direction = vector.to(activation.device)
                direction = direction / (direction.norm() + 1e-8)
                token_act = activation[batch_idx[valid], token_positions[valid]]
                proj = torch.einsum("bd,d->b", token_act, direction)
                activation[batch_idx[valid], token_positions[valid]] -= proj[:, None] * direction
            else:
                activation[batch_idx[valid], token_positions[valid]] += vector.to(activation.device)

        return activation

    scores = []

    for batch in dataloader:
        input_ids = batch["input_ids"].to(model.cfg.device)
        lengths = batch["lengths"].to(model.cfg.device)

        global batch_cache_start
        batch_cache_start = batch["cache_start"].to(model.cfg.device)

        if mode == "ablate":
            fwd_hooks = [
                (utils.get_act_name("resid_pre", layer_idx), localized_hook),
                (utils.get_act_name("resid_mid", layer_idx), localized_hook),
            ]
        else:
            fwd_hooks = [
                (utils.get_act_name("resid_pre", layer_idx), localized_hook),
            ]

        logits = model.run_with_hooks(
            input_ids,
            return_type="logits",
            fwd_hooks=fwd_hooks,
        )
        logits_last = get_last_token_logits(logits, lengths)
        scores.append(compute_refusal_logodds(logits_last, refusal_token_ids).detach().cpu())

        del logits
        torch.cuda.empty_cache()

    return torch.cat(scores, dim=0)


@torch.no_grad()
def scan_localized_intervention(
    dataloader,
    model,
    refusal_token_ids,
    vector,
    mode,   # "ablate" or "add"
):
    assert mode in {"ablate", "add"}

    n_layers = model.cfg.n_layers
    example_batch = next(iter(dataloader))
    n_positions = int(example_batch["post_instruction_length"])

    baseline_scores = compute_refusal_scores_with_hooks(
        dataloader=dataloader,
        model=model,
        refusal_token_ids=refusal_token_ids,
        fwd_hooks=[],
    )
    baseline_mean = baseline_scores.mean().item()
    baseline_cases = baseline_scores >= 0

    score_delta = torch.zeros((n_layers, n_positions))
    flip_count = torch.zeros((n_layers, n_positions), dtype=torch.long)

    for layer_idx in tqdm(range(n_layers), desc=f"localized-{mode}"):
        for rel_pos in range(n_positions):
            perturbed_scores = compute_localized_scores(
                dataloader=dataloader,
                model=model,
                refusal_token_ids=refusal_token_ids,
                vector=vector,
                layer_idx=layer_idx,
                rel_pos=rel_pos,
                mode=mode,
            )
            perturbed_cases = perturbed_scores >= 0

            score_delta[layer_idx, rel_pos] = perturbed_scores.mean().item() - baseline_mean

            if mode == "ablate":
                # harmful test: refusal -> non-refusal
                flips = baseline_cases & (~perturbed_cases)
            else:
                # harmless test: non-refusal -> refusal
                flips = (~baseline_cases) & perturbed_cases

            flip_count[layer_idx, rel_pos] = int(flips.sum().item())

    return {
        "baseline_scores": baseline_scores,
        "baseline_mean": baseline_mean,
        "baseline_cases": baseline_cases,
        "score_delta": score_delta,
        "flip_count": flip_count,
    }


def get_best_site_from_heatmap(score_delta, mode):
    if mode == "ablate":
        flat_idx = torch.argmin(score_delta)
    else:
        flat_idx = torch.argmax(score_delta)
    layer_idx, rel_pos = torch.unravel_index(flat_idx, score_delta.shape)
    return int(layer_idx), int(rel_pos)

In [ ]:
# harmful test: localized ablation (bypass-side)
localized_bypass = scan_localized_intervention(
    dataloader=dl_hf_test,
    model=model,
    refusal_token_ids=refusal_token_set,
    vector=refusal_direction,
    mode="ablate",
)

# harmless test: localized addition (induce-side)
localized_induce = scan_localized_intervention(
    dataloader=dl_hl_test,
    model=model,
    refusal_token_ids=refusal_token_set,
    vector=refusal_direction,
    mode="add",
)

best_bypass_layer, best_bypass_pos = get_best_site_from_heatmap(
    localized_bypass["score_delta"], mode="ablate"
)
best_induce_layer, best_induce_pos = get_best_site_from_heatmap(
    localized_induce["score_delta"], mode="add"
)

In [ ]:
plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

fig, axes = plt.subplots(2, 2, figsize=(12, 7), constrained_layout=True)


v0 = max(abs(localized_bypass["score_delta"].min().item()),
         abs(localized_bypass["score_delta"].max().item()))
v1 = max(abs(localized_induce["score_delta"].min().item()),
         abs(localized_induce["score_delta"].max().item()))
v = max(v0, v1)

im0 = axes[0, 0].imshow(
    localized_bypass["score_delta"].numpy(),
    aspect="auto",
    origin="lower",
    cmap="coolwarm",
    vmin=-v,
    vmax=v,
)
# axes[0, 0].scatter(
#     best_bypass_pos, best_bypass_layer,
#     s=60, facecolors="none", edgecolors="black", linewidths=1.5
# )
axes[0, 0].set_title("Ablation score Δ")
axes[0, 0].set_xlabel("Post-instruction token")
axes[0, 0].set_ylabel("Layer")

im1 = axes[0, 1].imshow(
    localized_induce["score_delta"].numpy(),
    aspect="auto",
    origin="lower",
    cmap="coolwarm",
    vmin=-v,
    vmax=v,
)
# axes[0, 1].scatter(
#     best_induce_pos, best_induce_layer,
#     s=60, facecolors="none", edgecolors="black", linewidths=1.5
# )
axes[0, 1].set_title("Addition score Δ")
axes[0, 1].set_xlabel("Post-instruction token")
axes[0, 1].set_ylabel("Layer")


im2 = axes[1, 0].imshow(
    localized_bypass["flip_count"].numpy(),
    aspect="auto",
    origin="lower",
    cmap="viridis",
)
# axes[1, 0].scatter(
#     best_bypass_pos, best_bypass_layer,
#     s=60, facecolors="none", edgecolors="white", linewidths=1.5
# )
axes[1, 0].set_title("Ablation flipped cases")
axes[1, 0].set_xlabel("Post-instruction token")
axes[1, 0].set_ylabel("Layer")

im3 = axes[1, 1].imshow(
    localized_induce["flip_count"].numpy(),
    aspect="auto",
    origin="lower",
    cmap="viridis",
)
# axes[1, 1].scatter(
#     best_induce_pos, best_induce_layer,
#     s=60, facecolors="none", edgecolors="white", linewidths=1.5
# )
axes[1, 1].set_title("Addition flipped cases")
axes[1, 1].set_xlabel("Post-instruction token")
axes[1, 1].set_ylabel("Layer")


cbar1 = fig.colorbar(im1, ax=axes[0, :], shrink=0.9)
cbar1.set_label("Δ refusal log-odds")

cbar2 = fig.colorbar(im3, ax=axes[1, :], shrink=0.9)
cbar2.set_label("Flipped cases")

save_path = SAVE_DIR / f"{model_name}-localized-intervention-figure.png"
plt.savefig(save_path, dpi=300, bbox_inches="tight", pad_inches=0.05)

plt.show()

print("best localized bypass site :", (best_bypass_layer, best_bypass_pos))
print("best localized induce site :", (best_induce_layer, best_induce_pos))
print("bypass baseline mean score :", localized_bypass["baseline_mean"])
print("induce baseline mean score :", localized_induce["baseline_mean"])

## 3-2. attention head ablation after proxy metric change

**Observation**

Layer **9** appears most influential.

**Question**

Is there a **single responsible attention head** producing refusal?

**Result**

- Heads **9.3** and **9.2** have strongest effects in layer 9.

- Removing either head alone **does not remove refusal.**

- Suggests refusal is implemented by a **distributed circuit across multiple heads,** not a single head.

In [ ]:
@torch.no_grad()
def collect_head_r_projections_last_token(
    dataloader,
    model,
    vector,
):
    device = model.cfg.device
    direction = vector.to(device)
    direction = direction / (direction.norm() + 1e-8)

    n_layers = model.cfg.n_layers
    hook_names = [utils.get_act_name("z", l) for l in range(n_layers)]

    proj_batches = []

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        lengths = batch["lengths"].to(device)

        _, cache = model.run_with_cache(
            input_ids,
            names_filter=lambda name: name in hook_names,
        )

        batch_size = input_ids.shape[0]
        batch_idx = torch.arange(batch_size, device=device)

        layer_head_vals = []

        for l in range(n_layers):
            z = cache[utils.get_act_name("z", l)]                  # [batch, seq, n_heads, d_head]
            z_last = z[batch_idx, lengths - 1, :, :]              # [batch, n_heads, d_head]

            head_out = torch.einsum(
                "bhd,hdk->bhk",
                z_last,
                model.W_O[l]
            )                                                     # [batch, n_heads, d_model]

            proj = torch.einsum("bhk,k->bh", head_out, direction) # [batch, n_heads]
            layer_head_vals.append(proj.detach().cpu())

        layer_head_vals = torch.stack(layer_head_vals, dim=1)     # [batch, n_layers, n_heads]
        proj_batches.append(layer_head_vals)

        del cache
        torch.cuda.empty_cache()

    return torch.cat(proj_batches, dim=0)


def summarize_head_projection_diff(harmful_proj, harmless_proj):
    harmful_mean = harmful_proj.mean(dim=0)      # [n_layers, n_heads]
    harmless_mean = harmless_proj.mean(dim=0)    # [n_layers, n_heads]
    diff = harmful_mean - harmless_mean
    return harmful_mean, harmless_mean, diff


def rank_heads_by_projection(diff):
    rows = []
    n_layers, n_heads = diff.shape

    for l in range(n_layers):
        for h in range(n_heads):
            rows.append({
                "layer": l,
                "head": h,
                "proj_diff": float(diff[l, h].item()),
            })

    df = pd.DataFrame(rows).sort_values("proj_diff", ascending=False).reset_index(drop=True)
    return df

In [ ]:
harmful_head_proj = collect_head_r_projections_last_token(
    dataloader=dl_hf_test,
    model=model,
    vector=refusal_direction,
)

harmless_head_proj = collect_head_r_projections_last_token(
    dataloader=dl_hl_test,
    model=model,
    vector=refusal_direction,
)

harmful_head_mean, harmless_head_mean, head_proj_diff = summarize_head_projection_diff(
    harmful_head_proj,
    harmless_head_proj,
)

head_proj_ranking_df = rank_heads_by_projection(head_proj_diff)
print(head_proj_ranking_df.head(10))

In [ ]:
plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)

v = max(abs(head_proj_diff.min().item()),
        abs(head_proj_diff.max().item()))

im = ax.imshow(
    head_proj_diff.numpy(),
    aspect="auto",
    origin="lower",
    cmap="coolwarm",
    vmin=-v,
    vmax=v,
)

top_heads = head_proj_ranking_df.head(3)

# for _, row in top_heads.iterrows():
#     l = int(row["layer"])
#     h = int(row["head"])

#     ax.scatter(
#         h,
#         l,
#         s=70,
#         facecolors="none",
#         edgecolors="black",
#         linewidths=1.5,
#     )

#     ax.text(
#         h + 0.1,
#         l + 0.1,
#         f"{l}.{h}",
#         fontsize=10,
#     )

ax.set_xlabel("Head")
ax.set_ylabel("Layer")
ax.set_title("Head output alignment with refusal direction r")

cbar = fig.colorbar(im, ax=ax, shrink=0.9)
cbar.set_label("Projection onto r")

save_path = SAVE_DIR / f"{model_name}-head-direction-score-heatmap.png"
plt.savefig(save_path, dpi=300, bbox_inches="tight", pad_inches=0.05)

plt.show()

In [ ]:
def build_remove_r_from_head_hook(model, vector, layer_idx, head_idx):
    device = model.cfg.device
    direction = vector.to(device)
    direction = direction / (direction.norm() + 1e-8)

    WO = model.W_O[layer_idx, head_idx].to(device)   # [d_head, d_model]
    wr = torch.matmul(WO, direction)                 # [d_head]
    wr_norm_sq = wr.pow(2).sum() + 1e-8

    def hook_fn(z, hook):
        # z: [batch, seq, n_heads, d_head]
        z = z.clone()
        z_head = z[:, :, head_idx, :]
        coeff = torch.einsum("bsd,d->bs", z_head, wr) / wr_norm_sq
        z[:, :, head_idx, :] = z_head - coeff[..., None] * wr[None, None, :]
        return z

    return hook_fn


@torch.no_grad()
def scan_head_ablation_effect(
    dataloader,
    model,
    refusal_token_ids,
    vector,
):
    baseline_scores = compute_refusal_scores_with_hooks(
        dataloader=dataloader,
        model=model,
        refusal_token_ids=refusal_token_ids,
        fwd_hooks=[],
    )
    baseline_mean = baseline_scores.mean().item()
    baseline_cases = baseline_scores >= 0

    n_layers = model.cfg.n_layers
    n_heads = model.cfg.n_heads

    score_delta = torch.zeros((n_layers, n_heads))
    flip_count = torch.zeros((n_layers, n_heads), dtype=torch.long)

    for layer_idx in tqdm(range(n_layers), desc="head-ablation"):
        for head_idx in range(n_heads):
            hook_fn = build_remove_r_from_head_hook(
                model=model,
                vector=vector,
                layer_idx=layer_idx,
                head_idx=head_idx,
            )
            perturbed_scores = compute_refusal_scores_with_hooks(
                dataloader=dataloader,
                model=model,
                refusal_token_ids=refusal_token_ids,
                fwd_hooks=[(utils.get_act_name("z", layer_idx), hook_fn)],
            )
            perturbed_cases = perturbed_scores >= 0

            score_delta[layer_idx, head_idx] = perturbed_scores.mean().item() - baseline_mean
            flips = baseline_cases & (~perturbed_cases)   # harmful refusal -> non-refusal
            flip_count[layer_idx, head_idx] = int(flips.sum().item())

    return {
        "baseline_scores": baseline_scores,
        "baseline_mean": baseline_mean,
        "baseline_cases": baseline_cases,
        "score_delta": score_delta,
        "flip_count": flip_count,
    }


def summarize_selected_heads(head_scan, selected_heads):
    rows = []
    baseline_refusal = int(head_scan["baseline_cases"].sum().item())
    for layer_idx, head_idx in selected_heads:
        delta = float(head_scan["score_delta"][layer_idx, head_idx].item())
        flips = int(head_scan["flip_count"][layer_idx, head_idx].item())
        rows.append({
            "head": f"{layer_idx}.{head_idx}",
            "mean_delta": delta,
            "flip_count": flips,
            "baseline_refusal_count": baseline_refusal,
        })
    return pd.DataFrame(rows)

In [ ]:
head_scan = scan_head_ablation_effect(
    dataloader=dl_hf_test,
    model=model,
    refusal_token_ids=refusal_token_set,
    vector=refusal_direction,
)

selected_head_df = summarize_selected_heads(
    head_scan=head_scan,
    selected_heads=[(9, 3), (9, 2)],
)

In [ ]:
plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

# symmetric color range for score delta
v = max(
    abs(head_scan["score_delta"].min().item()),
    abs(head_scan["score_delta"].max().item()),
)

im0 = axes[0].imshow(
    head_scan["score_delta"].numpy(),
    aspect="auto",
    origin="lower",
    cmap="coolwarm",
    vmin=-v,
    vmax=v,
)
# axes[0].scatter(
#     [3, 2],
#     [9, 9],
#     s=60,
#     facecolors="none",
#     edgecolors="black",
#     linewidths=1.5,
# )
# axes[0].text(3.12, 9.12, "9.3", fontsize=10)
# axes[0].text(2.12, 8.72, "9.2", fontsize=10)
axes[0].set_title("Head ablation score Δ")
axes[0].set_xlabel("Head")
axes[0].set_ylabel("Layer")

im1 = axes[1].imshow(
    head_scan["flip_count"].numpy(),
    aspect="auto",
    origin="lower",
    cmap="viridis",
)
# axes[1].scatter(
#     [3, 2],
#     [9, 9],
#     s=60,
#     facecolors="none",
#     edgecolors="white",
#     linewidths=1.5,
# )
# axes[1].text(3.12, 9.12, "9.3", fontsize=10, color="white")
# axes[1].text(2.12, 8.72, "9.2", fontsize=10, color="white")
axes[1].set_title("Head ablation flipped cases")
axes[1].set_xlabel("Head")
axes[1].set_ylabel("Layer")

cbar0 = fig.colorbar(im0, ax=axes[0], shrink=0.9)
cbar0.set_label("Δ refusal log-odds")

cbar1 = fig.colorbar(im1, ax=axes[1], shrink=0.9)
cbar1.set_label("Flipped cases")

save_path = SAVE_DIR / f"{model_name}-head-ablation-paper-figure.png"
plt.savefig(save_path, dpi=300, bbox_inches="tight", pad_inches=0.05)

plt.show()

print("baseline harmful mean score:", head_scan["baseline_mean"])
print()
print(selected_head_df.to_string(index=False))